# 00. Competition Constraints and Rules

- Parent issue: `#14`
- Status: `gate / constraints freeze (pre-Wave-B)`
- Snapshot date: `2026-04-20`
- Summary: Freeze the competition-facing contract set (base model identity, scoring / normalization contract, LoRA cap, submission zip layout) and name every blocker that still requires human action on the Kaggle UI.

This notebook is the **gate** for all downstream work. Wave B notebooks (01–10) must not consume values from this notebook until the `BLOCKED` rows are resolved by a human pasting the Kaggle rules text into the repo.


## Audience and Why It Matters

**Who reads this:** project leads, notebook authors (01–10), and reviewers who need one verified source of truth for the Kaggle evaluator contract before any training, eval harness, or packaging code is written.

**Why it matters:**

- A wrong base model ID silently invalidates every adapter we train (tokenizer, chat template, LoRA target modules, attention implementation all branch on it).
- A wrong scoring / normalization assumption produces local metrics that don't map to the leaderboard.
- A wrong submission zip layout makes the final artifact unacceptable.
- A wrong LoRA rank cap makes an otherwise-trained adapter non-compliant.

Treating community write-ups and plan_v0.2 hard-claims as authoritative would guarantee at least one of these failures. This notebook therefore distinguishes **frozen** (evidence in repo + konbu17 baseline is safe to provisionally adopt), **provisional** (safe working value, cap, or layout backed by konbu17 only), and **BLOCKED** (needs Kaggle rules text captured in repo).


## Decision / Hypothesis

**Hypothesis:** the Kaggle competition page, rules tab, and submission demo notebook contain the exact contract for (a) base model identity + load recipe, (b) answer normalization / scoring, (c) LoRA rank cap, (d) submission zip layout. Everything else in the repo is derived, provisional, or rumor.

**Decision rule for this notebook:**

- If a constraint has authoritative Kaggle rules text pasted into the repo → `frozen`.
- If a constraint only has konbu17 baseline evidence or repo docs → `provisional` (safe working default, must not be hard-coded as authoritative downstream).
- If a constraint has no authoritative text **and** no safe working default → `BLOCKED`. Wave B cannot finalize while any BLOCKED row remains.

**Non-goals (per `docs/execution/plans/issue-14-constraints-freeze.md`):** no training, no inference, no Kaggle API automation of downloads by default, no `src/` code changes, no notebook-01+ edits. This notebook is spec + probe + export only.


## Environment and Reproduction

- Python: 3.11+ (no exotic deps; only `subprocess`, `json`, `csv`, `datetime`, `pathlib`, `shutil`, and `pandas` for table rendering).
- Run from repo root (`jupyter nbconvert --to notebook --execute notebooks/00_competition_constraints_and_rules.ipynb`) or open directly in Jupyter.
- Kaggle CLI is **optional**. The probe never raises on a machine without `kaggle` on PATH or without `~/.kaggle/kaggle.json`.
- Network downloads are **opt-in only** via the `RUN_DOWNLOAD` flag (default `False`). The probe only reads competition metadata and file listings.
- Outputs (JSON + CSV + open-questions text) are written under `data/eval/`, which is kept in git via `data/eval/.gitkeep`.


In [ ]:
from __future__ import annotations

import csv
import json
import os
import platform
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

# Snapshot date for every fact recorded below. Change only on re-verification.
SNAPSHOT_DATE = "2026-04-20"

# Guard flag: the Kaggle API download path is destructive (writes to data/raw/)
# and requires that the human has accepted competition rules on the web UI.
# Leave False unless you have explicitly joined the competition and want to test.
RUN_DOWNLOAD = False

# Resolve repo root deterministically so nbconvert --execute works from any cwd.
# This notebook lives in <repo>/notebooks/, so the repo root is its parent.
NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "00_competition_constraints_and_rules.ipynb").exists():
    REPO_ROOT = NOTEBOOK_DIR.parent
elif (NOTEBOOK_DIR / "notebooks" / "00_competition_constraints_and_rules.ipynb").exists():
    REPO_ROOT = NOTEBOOK_DIR
else:
    # Fallback: walk up looking for a recognizable repo marker.
    candidate = NOTEBOOK_DIR
    for _ in range(5):
        if (candidate / "docs" / "execution").exists():
            break
        candidate = candidate.parent
    REPO_ROOT = candidate

DATA_EVAL_DIR = REPO_ROOT / "data" / "eval"
DATA_EVAL_DIR.mkdir(parents=True, exist_ok=True)

print(f"Snapshot date   : {SNAPSHOT_DATE}")
print(f"Repo root       : {REPO_ROOT}")
print(f"Data eval dir   : {DATA_EVAL_DIR}")
print(f"Python          : {sys.version.split()[0]} on {platform.platform()}")
print(f"RUN_DOWNLOAD    : {RUN_DOWNLOAD} (set True only after accepting rules in browser)")


In [ ]:
# Safe Kaggle CLI probe. Never raises; captures stdout+stderr and continues.
# Returns a dict we can pretty-print and inspect per call.

COMPETITION_SLUG = "nvidia-nemotron-model-reasoning-challenge"


def _run(cmd: list[str], timeout: int = 30) -> dict:
    """Run a command, return a plain-dict result (no exceptions propagate)."""
    result: dict = {
        "cmd": " ".join(cmd),
        "returncode": None,
        "stdout": "",
        "stderr": "",
        "error": None,
    }
    try:
        completed = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            check=False,
            timeout=timeout,
        )
        result["returncode"] = completed.returncode
        result["stdout"] = (completed.stdout or "").strip()
        result["stderr"] = (completed.stderr or "").strip()
    except FileNotFoundError as exc:
        result["error"] = f"FileNotFoundError: {exc}"
    except subprocess.TimeoutExpired as exc:
        result["error"] = f"TimeoutExpired: {exc}"
    except Exception as exc:  # noqa: BLE001 - we intentionally swallow so the notebook executes cleanly
        result["error"] = f"{type(exc).__name__}: {exc}"
    return result


kaggle_path = shutil.which("kaggle")
print(f"`kaggle` on PATH? {'yes -> ' + kaggle_path if kaggle_path else 'no'}")

probe_results: dict[str, dict] = {"kaggle_path": kaggle_path}

if kaggle_path is None:
    print("[probe] Skipping Kaggle CLI calls: CLI not installed in this environment.")
    print("[probe] This is expected on CI / minimal-dep machines. No action required.")
else:
    version = _run(["kaggle", "--version"])
    probe_results["version"] = version
    print(f"\n$ kaggle --version\n{version.get('stdout') or version.get('error')}")

    listing = _run(["kaggle", "competitions", "list", "-s", "nvidia-nemotron"])
    probe_results["competitions_list"] = listing
    print("\n$ kaggle competitions list -s nvidia-nemotron")
    if listing.get("error"):
        print(f"  [error] {listing['error']}")
    else:
        print(listing.get("stdout") or "<empty stdout>")
        if listing.get("stderr"):
            print(f"  [stderr] {listing['stderr']}")

    files = _run(["kaggle", "competitions", "files", "-c", COMPETITION_SLUG])
    probe_results["competitions_files"] = files
    print(f"\n$ kaggle competitions files -c {COMPETITION_SLUG}")
    if files.get("error"):
        print(f"  [error] {files['error']}")
    else:
        print(files.get("stdout") or "<empty stdout>")
        if files.get("stderr"):
            print(f"  [stderr] {files['stderr']}")


In [ ]:
# Detect 403 / authorization / rules-not-accepted style failures and print a loud STOP banner.
# Heuristic: scan combined stderr+stdout of every probe call for known indicators.
# This keeps the notebook executing cleanly on machines without the CLI (no banner printed).

AUTH_INDICATORS = (
    "403",
    "forbidden",
    "unauthorized",
    "must accept",
    "accept the competition rules",
    "please join",
    "you have not",
)


def _looks_like_auth_failure(result: dict | None) -> bool:
    if not result:
        return False
    blob = " ".join([str(result.get("stdout", "")), str(result.get("stderr", "")), str(result.get("error", ""))]).lower()
    return any(token in blob for token in AUTH_INDICATORS)


auth_failure_detected = any(
    _looks_like_auth_failure(probe_results.get(key))
    for key in ("competitions_list", "competitions_files")
)

if RUN_DOWNLOAD:
    # Opt-in path. This is the only place we attempt an actual data pull.
    # Guarded so nbconvert on a clean CI never downloads anything.
    dl_target = REPO_ROOT / "data" / "raw"
    dl_target.mkdir(parents=True, exist_ok=True)
    print(f"[opt-in] Attempting download to {dl_target} ...")
    download = _run([
        "kaggle",
        "competitions",
        "download",
        "-c",
        COMPETITION_SLUG,
        "-p",
        str(dl_target),
    ], timeout=300)
    probe_results["competitions_download"] = download
    if download.get("error"):
        print(f"  [error] {download['error']}")
    else:
        print(download.get("stdout") or "<empty stdout>")
        if download.get("stderr"):
            print(f"  [stderr] {download['stderr']}")
    if _looks_like_auth_failure(download):
        auth_failure_detected = True
else:
    print("[probe] RUN_DOWNLOAD=False -> skipping `kaggle competitions download`. Safe default.")

if auth_failure_detected:
    banner = "\n".join([
        "",
        "==============================================================================",
        "  STOP: KAGGLE AUTHORIZATION FAILURE (403 / rules not accepted)",
        "==============================================================================",
        "  The Kaggle CLI returned an auth-style error for this competition.",
        "  Until the rules are accepted in the browser, the CLI cannot list/download",
        "  competition files and this notebook cannot resolve any BLOCKED row.",
        "",
        "  Manual recovery steps (do these in order):",
        f"    1. Open https://www.kaggle.com/competitions/{COMPETITION_SLUG}",
        "    2. Click the Rules tab, scroll to the bottom, click 'I Understand and Accept'.",
        "    3. Confirm the page now says 'Joined' (top-right of the competition page).",
        "    4. Re-run this notebook OR re-run `kaggle competitions files -c " + COMPETITION_SLUG + "`.",
        "    5. Paste the rules / scoring / submission text into the repo (see checklist below).",
        "==============================================================================",
        "",
    ])
    print(banner)
else:
    print("[probe] No auth failure detected in probe output (or CLI not installed).")


## Manual Kaggle UI Steps & Rules-Capture Checklist

The Kaggle CLI exposes competition metadata (deadline, data files) but **does not** expose the rules-tab text, scoring contract, or submission demo load recipe. A human must open the Kaggle web UI, accept the rules, and paste the authoritative text into the repo before the `BLOCKED` rows below can be promoted to `frozen`.

### UI steps (in order)

1. Open <https://www.kaggle.com/competitions/nvidia-nemotron-model-reasoning-challenge>.
2. Click **Rules**. Scroll to the bottom. Click **I Understand and Accept**.
3. Confirm the page header now shows **Joined**.
4. Open the **Submission Demo** notebook linked from the competition page and save its exact model-load cell as plain text.
5. Open the **Evaluation** / **Scoring** section of the competition overview and save the scoring text verbatim.

### Rules-capture checklist (paste into repo under `docs/architecture/COMPETITION.md` with `snapshot_date: 2026-04-20`)

- [ ] **Base model identity** — exact HuggingFace slug (e.g. `nvidia/...`) plus revision / sha, as stated by the rules or the submission demo notebook.
- [ ] **Base-model load recipe** — the exact `AutoModelForCausalLM.from_pretrained(...)` call used by the demo: `trust_remote_code`, dtype (`bf16` / `fp16`), `attn_implementation`, `enable_thinking` / special-token handling, `pad_token` fallback.
- [ ] **Scoring / normalization contract** — does the scorer strip whitespace? Lowercase? Extract `\boxed{}`? Accept `<think>` text? Penalize reasoning in the answer field? Is it exact match, per-category, macro or micro?
- [ ] **LoRA rank cap** — authoritative rank ceiling (provisional working value: `r <= 32`).
- [ ] **Allowed LoRA target modules** — whitelist / blacklist if specified.
- [ ] **Adapter dtype** — must adapters be `bf16`? `fp16`? Either?
- [ ] **Max submission file size** — any cap on `adapter_model.safetensors` or the zip overall.
- [ ] **Submission zip layout** — required root contents (provisional: `adapter_config.json` + `adapter_model.safetensors` at root).
- [ ] **Extra required files** — metadata, manifest, README, license? Any forbidden filenames?
- [ ] **Per-day / per-team submission limits** and **final-submission selection rule**.
- [ ] **Deadlines with explicit timezone** (entry deadline, final deadline).

Each checkbox should link to the exact Kaggle URL fragment that sourced the text.


## Method and Outputs

The next cell builds the authoritative constraint table from `docs/execution/plans/issue-14-constraints-freeze.md` §4. Each row has:

- `variable` — the contract name downstream notebooks will reference.
- `status` — one of `frozen`, `provisional`, `BLOCKED`.
- `value` — the working value (or the string `BLOCKED` for unresolved rows; never a guessed slug).
- `evidence` — what we looked at (repo doc or konbu17 cell); never a rumor.
- `confidence` — `low` / `medium` / `high`.
- `manual_action` — exactly what a human must do on Kaggle to promote the row.

**Outputs written by this notebook:**

- `data/eval/competition_constraints.json`
- `data/eval/competition_constraints.csv`
- `data/eval/open_questions_snapshot.txt`


## Constraint Table


In [ ]:
# Constraint table, sourced verbatim from docs/execution/plans/issue-14-constraints-freeze.md section 4.
# Every entry is either evidence-backed (repo doc or konbu17 cell) or explicitly BLOCKED.
# We deliberately do NOT hardcode the base model ID, scoring contract, or \\boxed/<think> defaults.

CONSTRAINTS: list[dict] = [
    {
        "variable": "base_model_id",
        "status": "BLOCKED",
        "value": "BLOCKED",
        "evidence": (
            "docs/architecture/COMPETITION.md marks this unresolved; "
            "docs/planning/plan_v0.2.md hard-claims 'nvidia/NVIDIA-Nemotron-3-Nano-4B-BF16' but that is unverified; "
            "data/external/konbu17/cells/cell10.py uses KaggleHub 'metric/nemotron-3-nano-30b-a3b-bf16/transformers/default'. "
            "These disagree; neither is confirmed against Kaggle rules or the submission demo notebook."
        ),
        "confidence": "low",
        "manual_action": (
            "Open the Kaggle rules page and the submission demo notebook; record the exact HF slug plus "
            "revision/sha into docs/architecture/COMPETITION.md with snapshot_date=2026-04-20."
        ),
    },
    {
        "variable": "base_model_load_recipe",
        "status": "BLOCKED",
        "value": "BLOCKED",
        "evidence": (
            "plan_v0.2.md uses trust_remote_code=True, BF16, enable_thinking=True; "
            "konbu17/cells/cell10.py uses BF16, trust_remote_code=True, attn_implementation='eager', "
            "pad_token=eos_token fallback. These are compatible but neither is authoritative."
        ),
        "confidence": "low",
        "manual_action": (
            "Copy the exact model-load cell from the Kaggle submission demo into the repo and preserve it "
            "as the canonical recipe."
        ),
    },
    {
        "variable": "scoring_normalization_contract",
        "status": "BLOCKED",
        "value": "BLOCKED",
        "evidence": (
            "docs/architecture/ARCHITECTURE.md centers exact-match normalization; "
            "docs/planning/plan_v0.2.md hardcodes \\boxed{} extraction and format rewards; "
            "docs/analysis/ADVERSARIAL_REVIEW.md argues the task is rule inference with plain-string answers. "
            "No authoritative scoring text is in the repo."
        ),
        "confidence": "low",
        "manual_action": (
            "Paste the Kaggle Evaluation/Scoring text into docs/architecture/COMPETITION.md and resolve whether "
            "reasoning text is ignored, normalized, or rejected (and whether \\boxed{} extraction applies)."
        ),
    },
    {
        "variable": "lora_only_constraint",
        "status": "frozen",
        "value": "LoRA adapter only",
        "evidence": (
            "docs/architecture/COMPETITION.md explicitly states LoRA-adapter-only submissions; "
            "plan_v0.2.md frames the submission as a LoRA adapter."
        ),
        "confidence": "medium",
        "manual_action": (
            "Verify the rules page for any exception or extra artifact requirement."
        ),
    },
    {
        "variable": "lora_rank_cap",
        "status": "provisional",
        "value": "r <= 32",
        "evidence": (
            "docs/architecture/COMPETITION.md flags 'LoRA rank <= 32'; "
            "data/external/konbu17/cells/cell11.py uses r=32; "
            "docs/analysis/PLAN_V0_2_REVIEW_PLAN.md treats this as the safe working cap."
        ),
        "confidence": "medium",
        "manual_action": (
            "Confirm the cap on the Kaggle rules page before any rank sweep above 32."
        ),
    },
    {
        "variable": "lora_target_modules",
        "status": "provisional",
        "value": (
            "['q_proj','k_proj','v_proj','o_proj','in_proj','out_proj','up_proj','down_proj','lm_head']"
        ),
        "evidence": (
            "data/external/konbu17/cells/cell11.py uses this exact 9-module set (konbu17 baseline)."
        ),
        "confidence": "medium",
        "manual_action": (
            "Check whether the rules restrict target modules or whether a narrower subset is required."
        ),
    },
    {
        "variable": "adapter_dtype",
        "status": "provisional",
        "value": "bf16",
        "evidence": (
            "data/external/konbu17/cells/cell10.py loads BF16; "
            "plan_v0.2.md and COMPETITION.md describe BF16-era handling."
        ),
        "confidence": "medium",
        "manual_action": (
            "Verify whether the evaluator requires BF16, FP16, or accepts either."
        ),
    },
    {
        "variable": "submission_zip_layout",
        "status": "provisional",
        "value": "root contains only adapter_config.json and adapter_model.safetensors",
        "evidence": (
            "data/external/konbu17/cells/cell17.py zips exactly those two files at the zip root."
        ),
        "confidence": "high",
        "manual_action": (
            "Confirm whether Kaggle wants any extra metadata file or a different archive root."
        ),
    },
    {
        "variable": "provenance_placement",
        "status": "frozen",
        "value": "out-of-band only",
        "evidence": (
            "docs/architecture/ARCHITECTURE.md defines PackageManifest; konbu17 cell17.py keeps the submission zip minimal."
        ),
        "confidence": "high",
        "manual_action": (
            "Keep PackageManifest in docs / experiments, not inside the submission zip, unless rules say otherwise."
        ),
    },
    {
        "variable": "max_submission_file_size",
        "status": "BLOCKED",
        "value": "BLOCKED",
        "evidence": (
            "No authoritative limit is present in the repo; konbu17 baseline does not document a cap."
        ),
        "confidence": "low",
        "manual_action": (
            "Check the Kaggle rules / submission demo for any per-file or per-zip size limit."
        ),
    },
    {
        "variable": "competition_deadline",
        "status": "provisional",
        "value": "2026-06-15 23:59 (timezone unverified)",
        "evidence": (
            "docs/analysis/PLAN_V0_2_REVIEW_PLAN.md records deadline 2026-06-15 23:59 from kaggle CLI on 2026-04-20; "
            "timezone not exposed via CLI."
        ),
        "confidence": "medium",
        "manual_action": (
            "Confirm deadline timezone (UTC vs local) on the Kaggle competition page."
        ),
    },
]

# Render as a pandas DataFrame if available; fall back to plain text otherwise
# (keeps the notebook runnable on a minimal-deps machine).
try:
    import pandas as pd

    _df = pd.DataFrame(CONSTRAINTS)
    _have_pandas = True
except Exception as exc:  # noqa: BLE001
    print(f"[info] pandas unavailable ({exc}); rendering as plain text.")
    _df = None
    _have_pandas = False

if _have_pandas:
    try:
        from IPython.display import display
        display(_df)
    except Exception:
        print(_df.to_string(index=False))
else:
    for row in CONSTRAINTS:
        print(f"- {row['variable']:32s} [{row['status']:11s}] {row['value']}")


In [ ]:
# Export JSON + CSV to data/eval/ so downstream notebooks have a machine-readable contract.
# The JSON is the authoritative structured form; the CSV is for humans.

json_path = DATA_EVAL_DIR / "competition_constraints.json"
csv_path = DATA_EVAL_DIR / "competition_constraints.csv"

export_payload = {
    "snapshot_date": SNAPSHOT_DATE,
    "parent_issue": "#14",
    "generated_by": "notebooks/00_competition_constraints_and_rules.ipynb",
    "plan_doc": "docs/execution/plans/issue-14-constraints-freeze.md",
    "schema_version": "1.0",
    "rows": CONSTRAINTS,
}

json_path.write_text(json.dumps(export_payload, indent=2, sort_keys=False) + "\n", encoding="utf-8")
print(f"[export] wrote {json_path} ({json_path.stat().st_size} bytes)")

csv_fields = ["variable", "status", "value", "evidence", "confidence", "manual_action"]
with csv_path.open("w", encoding="utf-8", newline="") as fh:
    writer = csv.DictWriter(fh, fieldnames=csv_fields)
    writer.writeheader()
    for row in CONSTRAINTS:
        writer.writerow({k: row[k] for k in csv_fields})
print(f"[export] wrote {csv_path} ({csv_path.stat().st_size} bytes)")


## Frozen vs BLOCKED Variables


In [ ]:
frozen_rows = [r for r in CONSTRAINTS if r["status"] == "frozen"]
provisional_rows = [r for r in CONSTRAINTS if r["status"] == "provisional"]
blocked_rows = [r for r in CONSTRAINTS if r["status"] == "BLOCKED"]

print(f"Frozen      : {len(frozen_rows)}")
for r in frozen_rows:
    print(f"  - {r['variable']}: {r['value']}")

print(f"\nProvisional : {len(provisional_rows)}")
for r in provisional_rows:
    print(f"  - {r['variable']}: {r['value']}")

print(f"\nBLOCKED     : {len(blocked_rows)}")
for r in blocked_rows:
    print(f"  - {r['variable']}")
    print(f"      manual action: {r['manual_action']}")

if blocked_rows:
    print(
        "\n[gate] Wave B finalization CANNOT proceed until every BLOCKED variable above is "
        "resolved by a human pasting Kaggle rules / demo text into the repo."
    )
else:
    print("\n[gate] No BLOCKED variables. Wave B finalization may proceed.")


## Results / Open Risks

**Results (as of `2026-04-20`):**

- 2 rows are `frozen` (LoRA-only constraint, provenance placement).
- 5 rows are `provisional` (LoRA rank cap, target modules, adapter dtype, submission zip layout, deadline) — all safe konbu17-backed working values; none are hard-frozen.
- 4 rows are `BLOCKED` (base model id, load recipe, scoring/normalization contract, max submission file size). Wave B cannot finalize while any of these remain blocked.

**Open risks / known gaps (carry forward to issue #14 comment):**

- Kaggle CLI in this environment does not expose rules text, scoring semantics, or submission demo load code. Manual UI capture is required.
- Plan v0.2 hard-claims `nvidia/NVIDIA-Nemotron-3-Nano-4B-BF16` while konbu17 uses `metric/nemotron-3-nano-30b-a3b-bf16/...`. These are different models; at most one is correct. This notebook records the disagreement as evidence but does **not** pick a winner.
- Plan v0.2 hard-codes `\boxed{}` extraction and format rewards while architecture docs center exact-match. The scoring contract is `BLOCKED`; do not ship `\boxed{}`-centric normalization until rules text is captured.
- `RUN_DOWNLOAD` is False by default. A human with accepted rules must flip it and re-run locally to validate the download path.


In [ ]:
# Emit a dated, human-readable open-questions snapshot. This is the single file a
# human reviewer can look at to know what is still blocking Wave B.

open_questions_path = DATA_EVAL_DIR / "open_questions_snapshot.txt"

lines: list[str] = []
lines.append(f"# Open Questions Snapshot")
lines.append(f"# snapshot_date: {SNAPSHOT_DATE}")
lines.append(f"# generated_by: notebooks/00_competition_constraints_and_rules.ipynb")
lines.append(f"# parent_issue: #14")
lines.append("")
lines.append("## BLOCKED variables (Wave B gate)")
if blocked_rows:
    for r in blocked_rows:
        lines.append(f"- {r['variable']}")
        lines.append(f"    evidence       : {r['evidence']}")
        lines.append(f"    manual_action  : {r['manual_action']}")
else:
    lines.append("- (none)")

lines.append("")
lines.append("## Provisional variables (safe working values, not authoritative)")
for r in provisional_rows:
    lines.append(f"- {r['variable']} = {r['value']}")
    lines.append(f"    manual_action  : {r['manual_action']}")

lines.append("")
lines.append("## Kaggle UI follow-ups (human)")
lines.append("- Accept competition rules in browser and confirm 'Joined'.")
lines.append("- Paste Kaggle scoring text verbatim into docs/architecture/COMPETITION.md.")
lines.append("- Copy the submission demo model-load cell verbatim into docs/architecture/COMPETITION.md.")
lines.append("- Record any per-file / per-zip size limits.")
lines.append("- Record deadline timezone explicitly.")

open_questions_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
print(f"[export] wrote {open_questions_path} ({open_questions_path.stat().st_size} bytes)")
print("")
print("\n".join(lines))


## Sources

Snapshot date for every URL below: **`2026-04-20`**. Re-verify and bump the date when any source updates.

### Kaggle

- Competition page: <https://www.kaggle.com/competitions/nvidia-nemotron-model-reasoning-challenge>
- Rules tab: <https://www.kaggle.com/competitions/nvidia-nemotron-model-reasoning-challenge/rules>
- Discussion forum: <https://www.kaggle.com/competitions/nvidia-nemotron-model-reasoning-challenge/discussion>
- Submission demo (Ryan Holbrook): <https://www.kaggle.com/code/ryanholbrook/nvidia-nemotron-submission-demo>

### HuggingFace (candidate model cards, not yet authoritative)

- <https://huggingface.co/nvidia/NVIDIA-Nemotron-3-Nano-4B-BF16> (plan_v0.2 candidate)
- NVIDIA model hub (broader): <https://huggingface.co/nvidia>

### In-repo plan and analysis documents

- `docs/execution/plans/issue-14-constraints-freeze.md` (authoritative plan; constraint table section 4 is the source of this notebook's rows)
- `docs/planning/notebooks/00_competition_constraints_and_rules.md` (notebook-level plan)
- `docs/execution/ISSUE_REVIEW_HARNESS.md` (notebook review checklist)
- `docs/analysis/PLAN_V0_2_REVIEW_PLAN.md` (drift matrix)
- `docs/architecture/COMPETITION.md`
- `docs/architecture/ARCHITECTURE.md`
- `docs/planning/plan_v0.2.md`
- `docs/analysis/ADVERSARIAL_REVIEW.md`

### External baseline (konbu17)

- `data/external/konbu17/cells/cell10.py` (model load)
- `data/external/konbu17/cells/cell11.py` (LoRA config: r=32, 9-module target set)
- `data/external/konbu17/cells/cell13.py` (training loop reference)
- `data/external/konbu17/cells/cell17.py` (submission zip layout)
